<a href="https://colab.research.google.com/github/MegaDeadCowboy/jasper-burn-severity/blob/main/01_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data: NBAC perimeter + Sentinel-2 pre/post composites

Outputs (to `data/`, git-ignored):
- `nbac_jasper_2024.gpkg`: Jasper fire perimeter from NBAC 2024
- `s2_pre_2023.tif`, `s2_post_2025.tif`: SCL-masked, offset-corrected median composites (B04, B08, B8A, B12) on a 20 m UTM 11N grid covering the perimeter + 2 km buffer

In [1]:
!git clone https://github.com/MegaDeadCowboy/jasper-burn-severity.git
%cd jasper-burn-severity
!pip install -q pystac-client planetary-computer stackstac rioxarray geopandas

Cloning into 'jasper-burn-severity'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 17 (delta 3), reused 17 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 8.26 KiB | 8.26 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/jasper-burn-severity
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.0 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import urllib.request, zipfile
import numpy as np
import geopandas as gpd
import xarray as xr
import rioxarray  # noqa: F401  (registers .rio)
import pystac_client, planetary_computer, stackstac
import matplotlib.pyplot as plt
from shapely.geometry import Point

DATA = Path("data"); DATA.mkdir(exist_ok=True)
UTM = 32611            # UTM 11N
RES = 20               # m; 20 m grid for all bands
BUFFER_M = 2000        # unburned margin kept for validation
PRE  = "2023-07-01/2023-08-31"
POST = "2025-07-01/2025-08-31"
BANDS = ["B04", "B08", "B8A", "B12", "SCL"]
JASPER_TOWN = Point(-118.0814, 52.8737)  # lon, lat; used only to pick the fire polygon

## 1. NBAC perimeter
Single-year 2024 file from the 1972–2025 release (CWFIS Datamart).

In [ ]:
NBAC_URL = "https://cwfis.cfs.nrcan.gc.ca/downloads/nbac/NBAC_2024_20260513.zip"
nbac_zip = DATA / "NBAC_2024_20260513.zip"
if not nbac_zip.exists():
    urllib.request.urlretrieve(NBAC_URL, nbac_zip)
nbac_dir = DATA / "nbac_2024"
with zipfile.ZipFile(nbac_zip) as z:
    z.extractall(nbac_dir)
    print([n for n in z.namelist() if n.endswith((".shp", ".gdb/", ".gpkg"))])

In [ ]:
src = next(nbac_dir.rglob("*.shp"))   # if the release ships a .gdb instead, point gpd.read_file at it
nbac = gpd.read_file(src)
print(src.name, nbac.crs, len(nbac))
print(list(nbac.columns))

In [ ]:
# Polygons within 40 km of the townsite; inspect before choosing
town = gpd.GeoSeries([JASPER_TOWN], crs=4326).to_crs(nbac.crs)
near = nbac[nbac.distance(town.iloc[0]) < 40_000]
cols = [c for c in ["YEAR","NFIREID","ADMIN_AREA","NATPARK","NATPARKS","AG_SDATE","HS_SDATE","POLY_HA","ADJ_HA","GID"] if c in near.columns]
near[cols].sort_values("POLY_HA", ascending=False)

In [ ]:
# Largest polygon's NFIREID = the Jasper fire; dissolve any park/province pieces sharing that ID
fire_id = near.sort_values("POLY_HA", ascending=False).iloc[0]["NFIREID"]
perim = nbac[nbac["NFIREID"] == fire_id].dissolve(by="NFIREID").reset_index()
print("NFIREID:", fire_id, "| pieces:", (nbac["NFIREID"] == fire_id).sum(),
      "| POLY_HA sum:", round(nbac.loc[nbac["NFIREID"] == fire_id, "POLY_HA"].sum(), 1))
perim.to_file(DATA / "nbac_jasper_2024.gpkg")
perim.to_crs(4326).plot(edgecolor="k", facecolor="none");

## 2. Sentinel-2 L2A search (Planetary Computer STAC)

In [ ]:
perim_utm = perim.to_crs(UTM)
aoi_utm = perim_utm.buffer(BUFFER_M)
bounds = tuple(aoi_utm.total_bounds)          # in EPSG:32611
aoi_ll = aoi_utm.to_crs(4326).iloc[0]

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

def search(dt, max_cloud=60):
    items = catalog.search(collections=["sentinel-2-l2a"], intersects=aoi_ll,
                           datetime=dt, query={"eo:cloud_cover": {"lt": max_cloud}}).item_collection()
    print(dt, len(items), "items |", sorted({i.properties["s2:mgrs_tile"] for i in items}),
          "| baselines", sorted({i.properties.get("s2:processing_baseline") for i in items}))
    return items

items_pre, items_post = search(PRE), search(POST)

## 3. Stack, mask, correct, composite
- **SCL keep-list** `{2, 4, 5, 6, 7}`: dark area, vegetation, bare soil, water, unclassified. Class 2 stays in because Sen2Cor often labels fresh char as "dark area". Clouds, shadow, cirrus, snow, and saturated/no-data pixels are dropped.
- **Offset:** baseline ≥ 04.00 → reflectance = (DN − 1000) / 10000; earlier → DN / 10000.

In [ ]:
SCL_KEEP = [2, 4, 5, 6, 7]

def composite(items):
    da = stackstac.stack(items, assets=BANDS, epsg=UTM, resolution=RES, bounds=bounds,
                         dtype="float32", fill_value=np.float32(np.nan), rescale=False,
                         chunksize=1024)
    scl = da.sel(band="SCL")
    refl = da.sel(band=BANDS[:-1])
    baseline = da["s2:processing_baseline"].astype(float)   # stackstac carries item properties as coords
    offset = xr.where(baseline >= 4.0, 1000.0, 0.0).astype("float32")
    refl = ((refl - offset) / 10000.0).where(scl.isin(SCL_KEEP))
    refl = refl.clip(0, 1)  # negative after offset = sensor noise
    med = refl.median("time", skipna=True)
    n_valid = refl.sel(band="B8A").notnull().sum("time").rename("n_valid")
    return med, n_valid

pre, n_pre = composite(items_pre)
post, n_post = composite(items_post)
pre

In [ ]:
%%time
pre, n_pre = (x.compute() for x in (pre, n_pre))
post, n_post = (x.compute() for x in (post, n_post))

## 4. Checks

In [ ]:
for name, n, img in [("pre", n_pre, pre), ("post", n_post, post)]:
    inside = n.rio.write_crs(UTM).rio.clip(perim_utm.geometry, drop=False)
    print(f"{name}: valid obs/pixel inside perimeter  min={int(np.nanmin(inside))}  "
          f"median={float(np.nanmedian(inside)):.0f} | no-data pixels: {int((n == 0).sum())} | "
          f"B12 range {float(img.sel(band='B12').min()):.3f}-{float(img.sel(band='B12').max()):.3f}")

In [ ]:
# Quick-look false colour (B12, B8A, B04); diagnostic only, not a final figure
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
for ax, img, t in zip(axs, [pre, post], ["Pre 2023", "Post 2025"]):
    rgb = img.sel(band=["B12", "B8A", "B04"]).transpose("y", "x", "band")
    rgb = (rgb / rgb.quantile(0.98, dim=["y", "x"])).clip(0, 1)
    ax.imshow(rgb, extent=[bounds[0], bounds[2], bounds[1], bounds[3]])
    perim_utm.boundary.plot(ax=ax, color="yellow", lw=0.8)
    ax.set_title(t); ax.set_axis_off()
plt.tight_layout()

## 5. Save

In [ ]:
def save(img, n, path):
    out = img.assign_coords(band=list(img.band.values)).rio.write_crs(UTM)
    out.attrs.update(scl_keep=str(SCL_KEEP), units="reflectance")
    out.rio.to_raster(path, compress="deflate")
    n.astype("uint16").rio.write_crs(UTM).rio.to_raster(path.with_name(path.stem + "_nvalid.tif"), compress="deflate")
    print("wrote", path)

save(pre, n_pre, DATA / "s2_pre_2023.tif")
save(post, n_post, DATA / "s2_post_2025.tif")